In [2]:
from src.database import get_database

db = get_database()

In [2]:
from pathlib import Path
import polars as pl

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"

input_file  = DATA_DIR / "polars" / "mexico_city_covid_data_dates.parquet"
output_file = DATA_DIR / "mexico_city_covid_unmapped.csv"

df_lazy = pl.scan_parquet(input_file).drop("_id")

df_lazy.sink_csv(output_file)

rows = pl.scan_parquet(input_file).select(pl.len()).collect()[0, 0]
size_mb = output_file.stat().st_size / 1024 / 1024

print(f"Archivo generado: {output_file}")
print(f"Filas: {rows:,}")
print(f"Tamaño: {size_mb:.1f} MB")

Archivo generado: /home/netfoor/Documents/university/KDD/covid19/data/mexico_city_covid_unmapped.csv
Filas: 7,840,211
Tamaño: 946.3 MB


In [4]:
print(db.list_collection_names())
print(len(db.list_collection_names()))

['raw_2023_data', 'mexico_city_covid_data', 'covid_unified_data', 'raw_2026_data', 'raw_2025_data', 'raw_2022_data', 'raw_2024_data', 'raw_2021_data', 'raw_2020_data']
9


In [5]:
from pathlib import Path
import polars as pl
from pymongo.errors import BulkWriteError
import time

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"

BATCH_SIZE = 50_000

def load_to_mongo(lazy_df, collection, label):
    total = lazy_df.select(pl.len()).collect()[0, 0]
    inserted = 0
    t0 = time.time()

    for offset in range(0, total, BATCH_SIZE):
        batch = lazy_df.slice(offset, BATCH_SIZE).collect()
        docs = batch.to_dicts()
        try:
            collection.insert_many(docs, ordered=False)
            inserted += len(docs)
        except BulkWriteError as e:
            inserted += e.details.get("nInserted", 0)

        print(f"[{label}] {inserted:,} / {total:,}", end="\r")

    print(f"\n[{label}] Completado en {time.time()-t0:.1f}s — {inserted:,} documentos")


parquet_path = DATA_DIR / "polars" / "mexico_city_covid_data_dates.parquet"
df_unmapped = pl.scan_parquet(parquet_path).drop("_id").with_columns([
    pl.col("FECHA_DEF").cast(pl.String),
    pl.col("FECHA_INGRESO").cast(pl.String),
    pl.col("FECHA_SINTOMAS").cast(pl.String),
])

col_unmapped = db["mexico_city_covid_unmapped"]
col_unmapped.drop()
load_to_mongo(df_unmapped, col_unmapped, "unmapped")

csv_path = DATA_DIR / "mexico_city_covid_final_report.csv"
df_mapped = pl.scan_csv(csv_path, infer_schema_length=10_000)


col_mapped = db["mexico_city_covid_mapped"]
col_mapped.drop()
load_to_mongo(df_mapped, col_mapped, "mapped  ")

print(f"\nTotal unmapped: {col_unmapped.count_documents({}):,}")
print(f"Total mapped:   {col_mapped.count_documents({}):,}")

[unmapped] 7,840,211 / 7,840,211
[unmapped] Completado en 248.2s — 7,840,211 documentos
[mapped  ] 7,840,211 / 7,840,211
[mapped  ] Completado en 406.3s — 7,840,211 documentos

Total unmapped: 7,840,211
Total mapped:   7,840,211


In [3]:
print(db.list_collection_names())
print(len(db.list_collection_names()))

['raw_2023_data', 'mexico_city_covid_data', 'covid_unified_data', 'raw_2026_data', 'raw_2025_data', 'raw_2022_data', 'raw_2024_data', 'mexico_city_covid_unmapped', 'raw_2021_data', 'raw_2020_data', 'mexico_city_covid_mapped']
11
